# Demo Modul 9: Attention dan Transformer

**Capstone.** Modul 7 dan 8 membaca kalimat satu token demi satu token.
Di sini rekurensi diganti pembobotan yang bergantung isi — dan dua hal yang
sebelumnya gratis (urutan token dan batas kalimat) harus dikembalikan sendiri.

## Capaian demo

1. Menulis scaled dot-product attention sendiri dan mencocokkannya dengan PyTorch.
2. Menunjukkan peran $\sqrt{d_k}$ sebagai dua angka entropi, bukan sebagai klaim.
3. Membuktikan **dengan angka** bahwa tanpa penyandian posisi model invarian
   terhadap permutasi, dan tanpa mask keluaran bergantung pada jumlah bantalan.
4. Menyetarakan anggaran parameter terhadap LSTM Modul 8 sebelum melatih.
5. Membaca peta attention dan mengukur pertumbuhan biaya terhadap panjang.

In [ ]:
import copy
import math
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

BASE_SEED = 42
QUICK_MODE = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_N, VAL_N, EPOCHS = ((2_400, 600, 2) if QUICK_MODE
                          else (6_000, 1_500, 4))
SEEDS = [BASE_SEED] if QUICK_MODE else [BASE_SEED, BASE_SEED + 1, BASE_SEED + 2]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_device() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

seed_everything(BASE_SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'quick_mode': QUICK_MODE,
       'train': TRAIN_N, 'validation': VAL_N, 'epochs': EPOCHS,
       'seeds': SEEDS})

## 1. Scaled dot-product attention dari nol

$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}(QK^\top/\sqrt{d_k})V$.
Implementasi sendiri dahulu, baru dicocokkan dengan PyTorch.

In [ ]:
def attention(Q, K, V, mask=None, skala=True):
    """Q (..., nq, dk), K/V (..., nk, dk). mask True = posisi diabaikan."""
    d_k = Q.size(-1)
    skor = Q @ K.transpose(-2, -1)
    if skala:
        skor = skor / math.sqrt(d_k)
    if mask is not None:
        skor = skor.masked_fill(mask, float('-inf'))
    bobot = torch.softmax(skor, dim=-1)
    return bobot @ V, bobot

seed_everything(BASE_SEED)
n, d_k = 5, 8
Q, K, V = (torch.randn(1, n, d_k) for _ in range(3))
keluaran, bobot = attention(Q, K, V)

acuan = F.scaled_dot_product_attention(Q, K, V)
selisih = (keluaran - acuan).abs().max().item()
print('bentuk bobot   :', tuple(bobot.shape))
print('jumlah tiap baris:', bobot.sum(-1).flatten().tolist())
print(f'selisih terhadap PyTorch: {selisih:.2e}')
assert torch.allclose(bobot.sum(-1), torch.ones(1, n), atol=1e-6)
assert selisih < 1e-5

In [ ]:
def entropi_rerata(d_k, skala, n=64, seed=BASE_SEED):
    seed_everything(seed)
    Q, K, V = (torch.randn(1, n, d_k) for _ in range(3))
    _, w = attention(Q, K, V, skala=skala)
    h = -(w * torch.log(w.clamp_min(1e-12))).sum(-1)
    return h.mean().item()

maks = math.log(64)
baris = [{'d_k': d, 'skala': s, 'entropi': entropi_rerata(d, s),
          'entropi_maks': maks}
         for d in (8, 256) for s in (True, False)]
print(pd.DataFrame(baris).to_string(index=False))

**Interpretasi.** Entropi maksimum untuk $n=64$ adalah $\log 64\approx4{,}16$
(bobot seragam). Tanpa pembagian $\sqrt{d_k}$, entropi anjlok saat $d_k$ besar:
softmax nyaris one-hot, dan gradien yang mengalir lewatnya nyaris nol.
Pembagian itu bukan kosmetik.

## 2. Pipeline AG News

Sama persis dengan Modul 7--8 agar perbandingannya langsung. Vocabulary tetap
dibangun hanya dari subset latih.

In [ ]:
ROOT = Path('../../data/raw/ag_news')
if not ROOT.exists():
    ROOT = Path('data/raw/ag_news')
train_file = ROOT / 'train.csv'
if not train_file.exists():
    raise FileNotFoundError(
        f'{train_file} tidak ditemukan. Letakkan AG News CSV sesuai data/README.md')

kolom = ['label', 'judul', 'ringkasan']
data = pd.read_csv(train_file, names=kolom, header=None)
if not str(data.iloc[0]['label']).strip().isdigit():
    data = data.iloc[1:].reset_index(drop=True)
data['teks'] = data['judul'].astype(str) + ' ' + data['ringkasan'].astype(str)
data['y'] = data['label'].astype(int) - 1

idx_train, idx_val = train_test_split(
    np.arange(len(data)), train_size=TRAIN_N, test_size=VAL_N,
    stratify=data['y'].to_numpy(), random_state=BASE_SEED)
teks_train = data['teks'].to_numpy()[idx_train]
teks_val = data['teks'].to_numpy()[idx_val]
y_train = data['y'].to_numpy()[idx_train]
y_val = data['y'].to_numpy()[idx_val]

POLA = re.compile(r"[a-z0-9']+")
def tokenisasi(teks):
    return POLA.findall(str(teks).lower())

cacah = Counter(t for s in teks_train for t in tokenisasi(s))
kosakata = ['<pad>', '<unk>'] + [w for w, n_ in cacah.most_common() if n_ >= 2]
stoi = {w: i for i, w in enumerate(kosakata)}
itos = {i: w for w, i in stoi.items()}
V = len(kosakata)

MAKS = 60
def ke_indeks(daftar_teks, maks=MAKS):
    X = torch.zeros(len(daftar_teks), maks, dtype=torch.long)
    L = torch.zeros(len(daftar_teks), dtype=torch.long)
    for i, teks in enumerate(daftar_teks):
        token = [stoi.get(t, 1) for t in tokenisasi(teks)][:maks] or [1]
        X[i, :len(token)] = torch.tensor(token)
        L[i] = len(token)
    return X, L

X_train, L_train = ke_indeks(teks_train)
X_val, L_val = ke_indeks(teks_val)
ds_train = TensorDataset(X_train, L_train, torch.tensor(y_train))
ds_val = TensorDataset(X_val, L_val, torch.tensor(y_val))
print({'train': len(ds_train), 'validation': len(ds_val), 'vocabulary': V})
assert kosakata[:2] == ['<pad>', '<unk>']

## 3. Penyandian posisi dan mask

Sinusoidal, jadi tidak menambah satu pun parameter — anggaran tetap dapat
dibandingkan dengan LSTM. Mask PyTorch memakai konvensi `True = diabaikan`.

In [ ]:
def penyandian_posisi(maks_len, d):
    pos = torch.arange(maks_len).unsqueeze(1).float()
    i = torch.arange(0, d, 2).float()
    pembagi = torch.exp(-math.log(10000.0) * i / d)
    pe = torch.zeros(maks_len, d)
    pe[:, 0::2] = torch.sin(pos * pembagi)
    pe[:, 1::2] = torch.cos(pos * pembagi)
    return pe

PE = penyandian_posisi(256, 100)
print('bentuk PE:', tuple(PE.shape), '| rentang nilai:',
      f'{PE.min():.2f} .. {PE.max():.2f}')
print('PE identik untuk posisi berbeda?',
      bool(torch.allclose(PE[3], PE[7])))

fig, ax = plt.subplots(figsize=(7, 2.6))
im = ax.imshow(PE[:60].T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xlabel('posisi token'); ax.set_ylabel('dimensi')
ax.set_title('Penyandian posisi sinusoidal (60 posisi pertama)')
fig.colorbar(im, ax=ax, shrink=0.85); plt.tight_layout(); plt.show()

## 4. Satu classifier dengan dua sakelar

`pakai_posisi` dan `pakai_mask` sengaja dapat dimatikan. Keduanya adalah
ablasi yang nanti dilatih, bukan sekadar dibicarakan.

In [ ]:
D_MODEL, N_HEAD, FF, KELAS = 100, 4, 175, 4

class TransformerClassifier(nn.Module):
    def __init__(self, pakai_posisi=True, pakai_mask=True,
                 d=D_MODEL, nhead=N_HEAD, ff=FF, dropout=0.1):
        super().__init__()
        self.pakai_posisi, self.pakai_mask, self.d = pakai_posisi, pakai_mask, d
        self.embedding = nn.Embedding(V, d, padding_idx=0)
        self.layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=nhead, dim_feedforward=ff,
            dropout=dropout, batch_first=True)
        self.head = nn.Linear(d, KELAS)
        self.register_buffer('pe', penyandian_posisi(256, d), persistent=False)

    def masukan(self, X):
        h = self.embedding(X)
        if self.pakai_posisi:
            h = h + self.pe[:X.size(1)].unsqueeze(0)
        return h

    def forward(self, X, panjang):
        h = self.masukan(X)
        pad = (X == 0)                       # True = posisi diabaikan
        h = self.layer(h, src_key_padding_mask=pad if self.pakai_mask else None)
        if self.pakai_mask:
            m = (~pad).unsqueeze(-1).to(h.dtype)
            wakil = (h * m).sum(1) / m.sum(1).clamp_min(1.0)
        else:
            wakil = h.mean(1)                # ikut merata-ratakan bantalan
        return self.head(wakil)

class LSTMClassifier(nn.Module):
    def __init__(self, hidden=96, d=D_MODEL):
        super().__init__()
        self.embedding = nn.Embedding(V, d, padding_idx=0)
        self.encoder = nn.LSTM(d, hidden, batch_first=True)
        self.head = nn.Linear(hidden, KELAS)

    def forward(self, X, panjang):
        emb = self.embedding(X)
        packed = pack_padded_sequence(emb, panjang.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, (h_n, _) = self.encoder(packed)
        return self.head(h_n[-1])

seed_everything(BASE_SEED)
m = TransformerClassifier().eval()
with torch.no_grad():
    print('logit:', tuple(m(X_val[:4], L_val[:4]).shape))

## 5. Tiga probe berupa angka

Semua dijalankan pada model **belum terlatih**: yang diuji adalah arsitektur,
bukan hasil belajar.

In [ ]:
i = int(L_val.argmax())                 # satu contoh panjang
x1, l1 = X_val[i:i + 1], L_val[i:i + 1]
n_asli = int(l1[0])

g = torch.Generator().manual_seed(BASE_SEED)
urutan = torch.randperm(n_asli, generator=g)
x2 = x1.clone()
x2[0, :n_asli] = x1[0, urutan]          # acak token asli, bantalan tetap

baris = []
for posisi in (False, True):
    seed_everything(BASE_SEED)
    mdl = TransformerClassifier(pakai_posisi=posisi).eval()
    with torch.no_grad():
        d = (mdl(x1, l1) - mdl(x2, l1)).abs().max().item()
    baris.append({'penyandian_posisi': posisi, 'selisih_logit_permutasi': d})
probe1 = pd.DataFrame(baris)
print(probe1.to_string(index=False))
assert probe1.loc[0, 'selisih_logit_permutasi'] < 1e-4
assert probe1.loc[1, 'selisih_logit_permutasi'] > 1e-3

Tanpa penyandian posisi, mengacak seluruh urutan kata **tidak mengubah
satu pun logit**: attention memperlakukan kalimat sebagai himpunan. Tidak ada
galat, tidak ada peringatan.

In [ ]:
xb, lb = X_val[:8], L_val[:8]
xb_panjang = torch.cat([xb, torch.zeros(8, 20, dtype=torch.long)], dim=1)

baris = []
for pakai_mask in (False, True):
    seed_everything(BASE_SEED)
    mdl = TransformerClassifier(pakai_mask=pakai_mask).eval()
    with torch.no_grad():
        d = (mdl(xb, lb) - mdl(xb_panjang, lb)).abs().max().item()
    baris.append({'mask': pakai_mask, 'selisih_logit_bantalan': d})
probe2 = pd.DataFrame(baris)
print(probe2.to_string(index=False))
assert probe2.loc[0, 'selisih_logit_bantalan'] > 1e-3
assert probe2.loc[1, 'selisih_logit_bantalan'] < 1e-4

In [ ]:
seed_everything(BASE_SEED)
mdl = TransformerClassifier().eval()
h = mdl.masukan(x1)
pad = (x1 == 0)
with torch.no_grad():
    _, w_mask = mdl.layer.self_attn(h, h, h, key_padding_mask=pad,
                                    need_weights=True)
    _, w_polos = mdl.layer.self_attn(h, h, h, need_weights=True)

massa_mask = w_mask[0, :n_asli, n_asli:].sum().item()
massa_polos = w_polos[0, :n_asli, n_asli:].sum().item()
print(f'panjang asli {n_asli} dari {x1.size(1)} kolom')
print(f'massa bobot ke posisi bantalan, dengan mask : {massa_mask:.2e}')
print(f'massa bobot ke posisi bantalan, tanpa mask  : {massa_polos:.4f}')
assert massa_mask < 1e-6

## 6. Anggaran parameter: hitung dahulu, baru latih

In [ ]:
def parameter_layer(d, ff):
    return 4 * d * d + 9 * d + ff * (2 * d + 1)

def parameter_lstm(hidden, d=D_MODEL):
    return 4 * hidden * (d + hidden + 2)

acuan = parameter_lstm(96) + 96 * KELAS + KELAS      # encoder + head LSTM
head_trf = D_MODEL * KELAS + KELAS

def ff_terdekat(target, d=D_MODEL, batas=1024):
    return min(range(1, batas + 1),
               key=lambda f: abs(parameter_layer(d, f) + head_trf - target))

FF_CARI = ff_terdekat(acuan)
enc_teori = parameter_layer(D_MODEL, FF_CARI)
seed_everything(BASE_SEED)
enc_nyata = sum(p.numel() for p in
                nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=N_HEAD,
                                           dim_feedforward=FF_CARI,
                                           batch_first=True).parameters())
print(f'acuan LSTM H=96 (encoder+head): {acuan:,}')
print(f'f terdekat: {FF_CARI} | encoder rumus {enc_teori:,} '
      f'| encoder PyTorch {enc_nyata:,}')
total_trf = enc_teori + head_trf
print(f'Transformer encoder+head: {total_trf:,} '
      f'| selisih {100 * (total_trf - acuan) / acuan:+.2f}%')
assert enc_teori == enc_nyata
assert FF_CARI == FF
assert abs(total_trf - acuan) / acuan <= 0.01

In [ ]:
baris = []
for nhead in (1, 2, 4, 10):
    seed_everything(BASE_SEED)
    mdl = TransformerClassifier(nhead=nhead)
    baris.append({'n_head': nhead, 'd_k': D_MODEL // nhead,
                  'parameter_layer': sum(p.numel()
                                         for p in mdl.layer.parameters())})
print(pd.DataFrame(baris).to_string(index=False))
print('\nJumlah head mengubah ruang relasi, bukan jumlah parameter.')

## 7. Pelatihan terkendali

Protokol identik dengan Modul 8: Adam $10^{-3}$, batch 64, clipping 1,0,
norma gradien dicatat **sebelum** clipping, checkpoint dari validation loss.

In [ ]:
BATCH = 64

def buat_loader(ds, shuffle, seed, batch=BATCH):
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=generator)

@torch.no_grad()
def evaluasi(model, ds):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='sum')
    total_loss, benar = 0.0, 0
    for xb, lb, yb in buat_loader(ds, False, BASE_SEED, 256):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb, lb)
        total_loss += criterion(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(ds), benar / len(ds)

def bangun(nama, seed):
    seed_everything(seed)
    if nama == 'lstm-h96':
        return LSTMClassifier(96)
    return TransformerClassifier(pakai_posisi=nama != 'trf-tanpa-posisi',
                                 pakai_mask=nama != 'trf-tanpa-mask')

def jalankan(nama, seed):
    model = bangun(nama, seed).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    loader = buat_loader(ds_train, True, seed)
    history = {'val_loss': [], 'val_acc': [], 'grad_norm': []}
    best_loss, best_state, n_update = float('inf'), None, 0
    sync_device(); mulai = time.perf_counter()

    for _ in range(EPOCHS):
        model.train()
        for xb, lb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            criterion(model(xb, lb), yb).backward()
            norm = torch.sqrt(sum((p.grad.detach() ** 2).sum()
                                  for p in model.parameters()
                                  if p.grad is not None)).item()
            history['grad_norm'].append(norm)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); n_update += 1
        val_loss, val_acc = evaluasi(model, ds_val)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())

    sync_device(); durasi = time.perf_counter() - mulai
    model.load_state_dict(best_state)
    train_loss, _ = evaluasi(model, ds_train)
    val_loss, val_acc = evaluasi(model, ds_val)
    enc = model.layer if hasattr(model, 'layer') else model.encoder
    enc_n = sum(p.numel() for p in enc.parameters())
    total_n = sum(p.numel() for p in model.parameters())
    catatan = {
        'run_id': f'{nama}-s{seed}', 'module': 'M09', 'seed': seed,
        'arsitektur': 'lstm' if nama == 'lstm-h96' else 'transformer',
        'posisi': nama != 'trf-tanpa-posisi' and nama != 'lstm-h96',
        'mask': nama != 'trf-tanpa-mask',
        'd_model': D_MODEL, 'n_head': N_HEAD if nama != 'lstm-h96' else 0,
        'dim_feedforward': FF if nama != 'lstm-h96' else 0,
        'max_length': MAKS, 'train_size': len(ds_train),
        'val_size': len(ds_val), 'epochs': EPOCHS, 'n_updates': n_update,
        'encoder_parameters': enc_n,
        'encoder_head_parameters': enc_n + sum(p.numel()
                                               for p in model.head.parameters()),
        'total_parameters': total_n,
        'model_mib_fp32': 4 * total_n / 1024 ** 2,
        'train_loss': train_loss, 'val_loss': val_loss, 'val_accuracy': val_acc,
        'grad_norm_mean': float(np.mean(history['grad_norm'])),
        'seconds_per_epoch': durasi / EPOCHS, 'device': str(DEVICE),
        'notes': 'quick demo' if QUICK_MODE else 'full protocol',
    }
    return model, history, catatan

In [ ]:
KONFIG = ['lstm-h96', 'trf-tanpa-posisi', 'trf-tanpa-mask', 'trf-penuh']
hasil, riwayat, simpan = [], {}, {}
for seed in SEEDS:
    for nama in KONFIG:
        print(f'Melatih {nama}, seed={seed} ...')
        mdl, hist, row = jalankan(nama, seed)
        hasil.append(row); riwayat[(nama, seed)] = hist; simpan[nama] = mdl

tabel = pd.DataFrame(hasil)
print(tabel[['run_id', 'encoder_head_parameters', 'total_parameters',
             'val_loss', 'val_accuracy', 'grad_norm_mean',
             'seconds_per_epoch']].to_string(index=False))
assert tabel.groupby('seed')['n_updates'].nunique().max() == 1
assert len(tabel) == len(KONFIG) * len(SEEDS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
epoch = np.arange(1, EPOCHS + 1)
for nama in KONFIG:
    hist = riwayat[(nama, SEEDS[0])]
    ax[0].plot(epoch, hist['val_acc'], marker='o', label=nama)
    ax[1].plot(epoch, hist['val_loss'], marker='o', label=nama)
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('validation accuracy')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('validation loss')
for a in ax:
    a.legend(fontsize=8); a.grid(alpha=.3); a.set_xticks(epoch)
plt.tight_layout(); plt.show()

penuh = tabel.loc[tabel['run_id'].str.startswith('trf-penuh'), 'val_accuracy'].mean()
for nama in ('trf-tanpa-posisi', 'trf-tanpa-mask', 'lstm-h96'):
    v = tabel.loc[tabel['run_id'].str.startswith(nama), 'val_accuracy'].mean()
    print(f'{nama:>18}: {v:.4f}  (selisih dari trf-penuh {v - penuh:+.4f})')

## 8. Peta attention dan biaya panjang

In [ ]:
model = simpan['trf-penuh'].to('cpu').eval()
i = int((L_val >= 12).nonzero()[0])
x, l = X_val[i:i + 1], int(L_val[i])
token = [itos[int(t)] for t in x[0, :l]]
with torch.no_grad():
    h = model.masukan(x)
    _, w = model.layer.self_attn(h, h, h, key_padding_mask=(x == 0),
                                 need_weights=True)

sisa = w[0, :l, l:].sum().item()
print(f'massa bobot ke bantalan: {sisa:.2e} (harus nol)')
k = min(l, 16)
fig, ax = plt.subplots(figsize=(6.4, 5.4))
im = ax.imshow(w[0, :k, :k].numpy(), cmap='viridis')
ax.set_xticks(range(k)); ax.set_xticklabels(token[:k], rotation=90, fontsize=7)
ax.set_yticks(range(k)); ax.set_yticklabels(token[:k], fontsize=7)
ax.set_xlabel('key'); ax.set_ylabel('query')
ax.set_title('Bobot attention (rerata empat head)')
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

datar = w[0, :k, :k].flatten()
for idx in datar.topk(3).indices.tolist():
    print(f'  {token[idx // k]:>14s} -> {token[idx % k]:<14s} {datar[idx]:.3f}')

Peta attention adalah **petunjuk arah aliran informasi**, bukan penjelasan
sebab-akibat. Bobot tinggi tidak membuktikan token itu yang menentukan kelas.

In [ ]:
@torch.no_grad()
def waktu_forward(model, n, ulang=20, batch=64):
    model = model.to(DEVICE).eval()
    x = torch.randint(2, V, (batch, n), device=DEVICE)
    l = torch.full((batch,), n, dtype=torch.long)
    model(x, l); sync_device()
    mulai = time.perf_counter()
    for _ in range(ulang):
        model(x, l)
    sync_device()
    return 1000 * (time.perf_counter() - mulai) / ulang

baris = []
for n in (30, 60, 120):
    baris.append({'n': n,
                  'lstm_ms': waktu_forward(simpan['lstm-h96'], n),
                  'transformer_ms': waktu_forward(simpan['trf-penuh'], n)})
biaya = pd.DataFrame(baris)
biaya['rasio_lstm'] = biaya['lstm_ms'] / biaya['lstm_ms'].iloc[0]
biaya['rasio_trf'] = biaya['transformer_ms'] / biaya['transformer_ms'].iloc[0]
print(biaya.to_string(index=False))
print('\nn dilipatduakan dua kali: rekuren ~O(n), attention ~O(n^2).')

In [ ]:
output = Path('M09_demo_metrics.csv')
tabel.to_csv(output, index=False)
print(f'{len(tabel)} baris disimpan ke {output}')

## Exit ticket

1. Apa yang terjadi pada logit bila urutan token diacak dan penyandian posisi dimatikan?
2. Di dua tempat mana mask harus bekerja, dan apa gejala bila salah satu terlewat?
3. Mengapa menambah head tidak menambah parameter?
4. Pada $n$ berapa Transformer mulai kalah cepat dari LSTM di perangkat Anda?

**Tugas capstone:** kerjakan `starter-mahasiswa.ipynb` — dua belas run
(empat konfigurasi kali tiga seed), termasuk kedua ablasi yang benar-benar dilatih.